# OEV Colab training

Runtime > Change runtime type > T4 GPU. Then run cells top to bottom.

The backbone cells fine-tune DeBERTa-v3-small (~540MB download first time) — the Laya-class jump.

In [ ]:
%cd /content
!rm -rf oev
!git clone https://github.com/divyanshudhruv/oev.git
%cd /content/oev
!pip install -q -e ".[dev,data,backbone]"

In [ ]:
!python -m pytest -q

## 1. Synthetic domain (from-scratch char model)

In [ ]:
!python -m oev.data_gen

In [ ]:
!python -m oev.train --preset tiny --epochs 4 --batch-size 256

In [ ]:
!python -m oev.evaluate --checkpoint checkpoints/oev-tiny.pt

## 2. Public benchmarks (from-scratch char model)

AG News and emotion converted to OEV format. Test splits stay sealed from training.

In [ ]:
!python -m oev.convert

In [ ]:
!python -m oev.train --preset tiny --epochs 2 --batch-size 128 --data-dir data/ag_news --out checkpoints_ag

In [ ]:
!python -m oev.benchmark --checkpoint checkpoints_ag/oev-tiny.pt --data-dir data/ag_news

## 3. Backbone fine-tune (Laya recipe) - THE REAL RUN

DeBERTa-v3-small already knows English; fine-tuning teaches only the decision task. Expect 0.90+ on AG News vs the ~0.29 from-scratch result.

In [ ]:
!python -m oev.train --backbone microsoft/deberta-v3-small --epochs 2 --batch-size 32 --data-dir data/ag_news --out checkpoints_bb

In [ ]:
!python -m oev.benchmark --checkpoint checkpoints_bb/oev-tiny.pt --data-dir data/ag_news

In [ ]:
!python -m oev.train --backbone microsoft/deberta-v3-small --epochs 2 --batch-size 32 --data-dir data/emotion --out checkpoints_bb_em

In [ ]:
!python -m oev.benchmark --checkpoint checkpoints_bb_em/oev-tiny.pt --data-dir data/emotion

## 4. Try the decide() API

In [ ]:
from oev.infer import OEV
agent = OEV("checkpoints_bb/oev-tiny.pt", device="cuda")
print(agent.decide("Wall St. Bears Claw Back Into the Black (Reuters)", {
    "topic": {"type": "choice", "options": ["world", "sports", "business", "sci/tech"], "instructions": "Which topic does this news article belong to?"},
}))
print(agent.decide("We were charged twice for the same order.", {
    "department": {"type": "choice", "options": ["billing", "technical", "sales", "other"]},
    "refund_requested": {"type": "noul"},
    "severity": {"type": "score", "levels": [1, 2, 3, 4, 5]},
}))

## 5. Save results to Drive (recommended)
Mount Drive first, then checkpoints survive Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/oev
!cp -r checkpoints* /content/drive/MyDrive/oev/